# OneVoice V2 — denoiser A/B
Use `MyDrive/onevoice_audio_v1`. Passthrough and DeepFilterNet run by default. RNNoise requires a real native `module:callable` adapter.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(WORK_ROOT / 'model_cache/huggingface')
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'PyYAML', 'soundfile', 'librosa', 'scipy', 'torch', 'sherpa-onnx', 'huggingface_hub', 'deepfilternet'], check=True)
MANIFEST = MYDRIVE / 'onevoice_audio_v1/manifest.jsonl'
if not MANIFEST.is_file(): raise FileNotFoundError('Run colab_data_audit_v2.ipynb first to create/recover manifest.jsonl')
REPORT_ROOT = WORK_ROOT / 'reports/denoiser'
CANDIDATES = ('passthrough', 'deepfilter')
print('Source:', REPO, '| Data:', MANIFEST, '| Reports:', REPORT_ROOT)


In [ ]:
for backend in CANDIDATES:
    for audio in ('clean', 'noisy'):
        subprocess.run([sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction', 'vi2en', '--split', 'test', '--audio', audio, '--denoiser', backend, '--report-dir', str(REPORT_ROOT / backend / audio)], check=True)


In [ ]:
for backend in CANDIDATES:
    if backend == 'passthrough': continue
    subprocess.run([sys.executable, 'scripts/evaluate_denoiser_gate.py', '--baseline-clean', str(REPORT_ROOT / 'passthrough/clean/aggregate.json'), '--baseline-noisy', str(REPORT_ROOT / 'passthrough/noisy/aggregate.json'), '--candidate-clean', str(REPORT_ROOT / backend / 'clean/aggregate.json'), '--candidate-noisy', str(REPORT_ROOT / backend / 'noisy/aggregate.json'), '--report', str(REPORT_ROOT / backend / 'gate.json')], check=False)


In [ ]:
import json
{f'{backend}/{audio}': json.loads((REPORT_ROOT / backend / audio / 'aggregate.json').read_text(encoding='utf-8')) for backend in CANDIDATES for audio in ('clean','noisy')}
